# 00 Getting started

This notebook does three things:

1. checks what your machine can do in this workshop,
2. introduces PIPS-IPM++ and the concept its interface is built on,
3. writes down the sample model that every other notebook uses.

There is nothing to build or solve here yet. That starts in 01.

## 1. Technical checks

Everything in the workshop installs on Linux, macOS and Windows. Two parts do not run everywhere:

| | Linux x86_64 and WSL | macOS | Windows |
|---|---|---|---|
| build models, export them, visualize and judge their structure | yes | yes | yes |
| find a structure with the regex method | yes | yes | yes |
| find a structure with the hypergraph method (mt-KaHyPar) | yes | yes | no, use WSL |
| solve with PIPS-IPM++ | yes, with the `solver` extra | no, use WSL or a cluster | no, use WSL or a cluster |

Solving needs MPI, even on a single process. MPI is an optional part of the install, because the
MPI runtime on PyPI does not exist for Windows. On Linux you opt in by starting JupyterLab with

```console
$ uv run --extra solver jupyter lab        # uv
$ pixi run -e solver lab                   # pixi
```

Run the cell below. It lists the installed packages and tells you what applies to your machine.

In [ ]:
import workshop

workshop.check()

Cells that need something your machine does not have start with a line like

```
%%requires solver
```

On a machine that has it, the cell runs as normal. On one that does not, it is skipped with a note
instead of failing, so every notebook runs top to bottom. The text around a skipped cell says what
it would have shown.

<details><summary>💡 Hint: I am on macOS or Windows. Is it worth continuing?</summary>

Yes. Building a model, exporting it, looking at its matrix and finding a good block structure all
run everywhere, and every notebook puts that part first. Only the solve at the end needs Linux. If
you have WSL or a Linux cluster, clone the workshop there as well and run the skipped cells; the
same `uv sync` or `pixi install` works unchanged.
</details>

<details><summary>💡 Hint: the check says "the solver extra is not installed" on Linux</summary>

JupyterLab was started from the default environment, which has no MPI. Stop it and start it again
with `uv run --extra solver jupyter lab` or `pixi run -e solver lab`. With uv, a plain `uv sync`
removes the extra again, so pass `--extra solver` whenever you sync.
</details>

On a machine with the solver, the next cell reports which solver library was loaded:

In [ ]:
%%requires solver
import pipsipmpp

for key, value in pipsipmpp.libpips_info().items():
    print(f"{key:16s} {value}")

| field | |
|---|---|
| `path`, `source` | which `libpips-ipmpp.so` was loaded. `bundled` is the one inside the package |
| `mpi_abi` | which MPI the library was built for. The `solver` extra installs Open MPI for you |
| `solver_version`, `built_against` | the solver, and the version the Python side was compiled for. Major and minor have to match, or the import fails early rather than crashing later |
| `solver_git_hash` | the exact sources. Record this next to any number you publish |

`libpips_info()` is also the first thing to print when a solve behaves oddly.

## 2. PIPS-IPM++

[PIPS-IPM++](https://gitlab.com/pips-ipmpp/pips-ipmpp) is an interior-point solver for linear
programs. It is built for problems that are too large to solve comfortably in one piece but that
consist of many parts which are only loosely coupled. Energy system models are the typical case:
a year of hourly dispatch is thousands of nearly independent hours, tied together by a few
investment decisions and a few annual constraints.

PIPS-IPM++ exploits that. It splits the problem into blocks, hands the blocks to different MPI
processes, and works on them at the same time. Each process only ever holds its own blocks, so the
problem can be larger than the memory of one machine.

### The fundamental concept: every variable gets assigned to a block

PIPS-IPM++ does not automatically search your model for structure. You state it, by assigning every 
variable to a block:

```
0      root   shared by every block   investment capacity, a shared budget
1..N   leaf   local to one block      dispatch in one hour, one region, one scenario
```

Everything else follows from that. The constraints (rows) can be automatically classifed based on the variables (columns):

- a constraint that touches the variables of only one leaf is local to that leaf,
- a constraint that touches the root and one leaf is still local to that leaf; the root variables form the
  left border, which every leaf may use,
- a constraint that touches two or more leaves is a linking constraint, and forms the bottom border.

Ordered by block, the constraint matrix then has this shape, called doubly-bordered
block-diagonal:

```
        root      leaf 1    leaf 2    leaf 3         variables
      +--------+---------+---------+---------+
 root |   A0   |         |         |         |   constraints on root variables only
      +--------+---------+---------+---------+
leaf 1|   A1   |   B1    |         |         |   constraints local to block 1
      +--------+---------+---------+---------+
leaf 2|   A2   |         |   B2    |         |   constraints local to block 2
      +--------+---------+---------+---------+
leaf 3|   A3   |         |         |   B3    |   constraints local to block 3
      +--------+---------+---------+---------+
 link |  Bl0   |   Bl1   |   Bl2   |   Bl3   |   constraints spanning several blocks
      +--------+---------+---------+---------+
       ^ left border                  ^ diagonal blocks, one per rank
```

The left border (A) is where local constraints reach into the root, the bottom border (Bl) holds
the linking constraints, and the diagonal blocks (B) are what is actually solved in parallel, using
one or more blocks per MPI rank. Inequalities have the same structure under the names C, D and Dl.

This structure suits a temporal decomposition of energy system optimization models well: investment 
decisions are root variables because every time step uses them, the dispatch in one time slice is 
a leaf block, and a yearly budget or emission cap is a linking constraint.

One kind of linking constraint is cheap: a constraint that touches only two neighbouring leaves, such 
as a storage level that carries over from the last hour of block 1 to the first hour of block 2.
PIPS-IPM++ calls these 2-link constraints and treats them much more efficiently than constraints 
that touch every block.

### How to assign blocks

You rarely write the full `var_block` vector yourself. Each modelling framework in this workshop 
has its own way of assignment:

| framework | how a variable gets its block | notebook |
|---|---|---|
| linopy, PyPSA | split a dimension such as `snapshot` into `n_blocks` slices | [01a](01a-linopy.ipynb), [02](02-pypsa.ipynb) |
| Pyomo | a `Suffix` that maps each variable to its block | [01b](01b-pyomo.ipynb) |
| PyOptInterface | a `block=` argument on every variable | [01c](01c-pyoptinterface.ipynb) |
| GAMSPy | the GAMS `.stage` attribute | [01d](01d-gamspy.ipynb) |
| a model file | found for you from the names or from the matrix structure | [03](03-structure-detection.ipynb) |
| your own matrices | the `var_block` array directly | [04](04-deep-dive.ipynb) |

## 3. The sample model

Every notebook uses the same small capacity expansion model, so you can compare them side by side:

> Invest in generation capacity, dispatch it hour by hour to meet demand, and stay under an
> emission budget.

Sets and data:

| symbol | meaning |
|---|---|
| $t \in \{0, \dots, T-1\}$ | hours, $T = 24$ to start with |
| $g \in G$ | technologies, `wind` and `gas` to start with |
| $\mathrm{demand}_t$ | demand in hour $t$ |
| $\mathrm{avail}_{g,t}$ | available share of capacity, a profile for wind and 1 for gas |
| $\mathrm{inv}_g, \mathrm{var}_g$ | investment cost per unit of capacity, cost per unit generated |
| $\mathrm{emis}_g$ | emissions per unit generated |
| $\mathrm{budget}$ | the emission budget for all hours together |

The model:

$$
\begin{aligned}
\min_{\mathrm{cap} \ge 0,\ \mathrm{gen} \ge 0} \quad
  & \sum_{g} \mathrm{inv}_g \, \mathrm{cap}_g + \sum_{g,t} \mathrm{var}_g \, \mathrm{gen}_{g,t} \\
\text{subject to} \quad
  & \sum_{g} \mathrm{gen}_{g,t} = \mathrm{demand}_t
    && \forall t && \text{(balance)} \\
  & \mathrm{gen}_{g,t} \le \mathrm{avail}_{g,t} \, \mathrm{cap}_g
    && \forall g, t && \text{(limit)} \\
  & \sum_{g,t} \mathrm{emis}_g \, \mathrm{gen}_{g,t} \le \mathrm{budget}
    && && \text{(carbon)}
\end{aligned}
$$

### Its block structure

Split the hours into $N$ contiguous slices, one per leaf block:

| | depends on | block |
|---|---|---|
| $\mathrm{cap}_g$ | no hour, it is shared by every hour | root, `0` |
| $\mathrm{gen}_{g,t}$ | one hour | the leaf that holds hour $t$ |
| $\mathrm{balance}_t$ | $\mathrm{gen}_{\cdot,t}$ of one hour | local to that leaf |
| $\mathrm{limit}_{g,t}$ | $\mathrm{gen}_{g,t}$ and the root column $\mathrm{cap}_g$ | local to that leaf, $\mathrm{cap}_g$ sits in the left border |
| carbon | $\mathrm{gen}$ of every hour | linking constraint, the bottom border |

With 24 hours, 2 technologies and 4 blocks that is 50 variables: 2 in the root and 12 in each leaf,
and one linking constraint out of 73. You will see exactly this picture in 01.

### Extensions used in the exercises

The 01 notebooks ask you to change the model and watch the matrix change. The extensions are:

- more hours, $T = 48$ or $T = 168$;
- another technology, for example `solar` with a daily availability profile;
- a storage with energy capacity $\mathrm{cap}^{s}$ (root), charging $\mathrm{ch}_t$, discharging
  $\mathrm{dis}_t$ and state of charge $\mathrm{soc}_t$ (leaf of hour $t$):

$$
\begin{aligned}
  & \sum_{g} \mathrm{gen}_{g,t} + \mathrm{dis}_t - \mathrm{ch}_t = \mathrm{demand}_t
    && \forall t && \text{(balance, replaces the one above)} \\
  & \mathrm{soc}_t = \mathrm{soc}_{t-1} + \eta \, \mathrm{ch}_t - \mathrm{dis}_t,
    \quad \mathrm{soc}_{-1} = 0
    && \forall t && \text{(storage level)} \\
  & \mathrm{soc}_t \le \mathrm{cap}^{s}
    && \forall t && \text{(storage limit)}
\end{aligned}
$$

and the objective gains $\mathrm{inv}^{s} \, \mathrm{cap}^{s}$.

Before you open 01, think about where the storage equations end up. Most of them are local to a leaf.
Which ones are not, how many are there for $N$ blocks, and what kind of linking row are they?

<details><summary>💡 Answer</summary>

The storage level in the first hour of a block refers to $\mathrm{soc}_{t-1}$, the last hour of the
previous block. Those equations touch two neighbouring leaves: $N - 1$ of them, all 2-link. All other
storage equations are local to their leaf, and $\mathrm{cap}^{s}$ joins the root.
</details>

## Next

Pick one of the 01 notebooks, ideally the framework you already use. You only need one, but feel free
to explore the others as well to get a feeling for their model formulation.

| you model in | open |
|---|---|
| linopy | [01a linopy](01a-linopy.ipynb) |
| Pyomo | [01b Pyomo](01b-pyomo.ipynb) |
| PyOptInterface | [01c PyOptInterface](01c-pyoptinterface.ipynb) |
| GAMS or GAMSPy | [01d GAMSPy](01d-gamspy.ipynb) |
| PyPSA, or none of these | [01a linopy](01a-linopy.ipynb), then [02 PyPSA](02-pypsa.ipynb) |